In [1]:
import pandas as pd
import numpy as np
import warnings
from micom.qiime_formats import load_qiime_medium
from micom.workflows import grow, build
from micom.interaction import interactions
from micom.workflows import load_results, GrowthResults
import sys
from itertools import product

sys.path.append('..')
from utils import analysis

warnings.filterwarnings('ignore')

## Preprocess normalized abundance dataframe to meet MICOM's requirement and generate metabolic models for each sample

In [ ]:
reference_taxonomy = pd.read_csv('./processed/reference_taxonomy.csv').drop(columns=['Unnamed: 0'])
agora_genomes = pd.read_csv('./processed/genome_size_ungapped.csv').drop(columns=['Unnamed: 0'])
reference_taxonomy_counts = pd.merge(reference_taxonomy, agora_genomes, on='id', how='right')
reference_taxonomy_counts['NCBI Taxonomy ID'] = reference_taxonomy_counts['NCBI Taxonomy ID'].astype(int).astype(str)
abundance = pd.read_csv('./raw/goll_agora_map.counts.csv')
abundance['taxon_id'] = abundance['taxon_id'].astype(str)

cols_to_keep = ['taxon_id'] + [col for col in abundance.columns if col.startswith('D') or col.endswith('-0') or col.endswith('-6') ]
abundance = abundance[cols_to_keep]
abundance = abundance.merge(reference_taxonomy_counts, left_on='taxon_id', right_on='NCBI Taxonomy ID', how='left')
abundance.set_index('taxon_id', inplace=True)
meta_cols = [
    'id', 'NCBI Taxonomy ID', 'kingdom', 'phylum', 'class', 'order',
    'family', 'genus', 'species', 'RefSeq', 'organism_name', 'genome_size_ungapped'
]

abundance['genome_size_ungapped'] = pd.to_numeric(abundance['genome_size_ungapped'], errors='coerce')
data_cols = abundance.columns.difference(meta_cols)
abundance[data_cols] = abundance[data_cols].div(abundance['genome_size_ungapped'], axis=0)
abundance = abundance.drop(columns=['RefSeq', 'organism_name', 'genome_size_ungapped', 'NCBI Taxonomy ID']).reset_index()
abundance.drop(columns=['kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'id']).set_index('taxon_id').to_csv('./processed/abundance_normalized_table.csv')
abundance = abundance.drop(columns=['id']).melt(id_vars=['taxon_id', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species'], var_name='sample_id', value_name='abundance').rename(columns={'taxon_id': 'id'})
abundance['sample_id'] = abundance['sample_id'].astype(str)
abundance.to_csv('./processed/abundance_normalized.csv')
abundance

In [ ]:
metadata = build(abundance, 
                 model_db='./raw/agora_species_103.qza',
                 out_folder='./models',
                 cutoff=0.0001,
                 threads=12)

In [ ]:
western = load_qiime_medium('./raw/western_diet_gut_agora.qza')
res = grow(manifest=metadata,
           medium=western,
           model_folder='./models',
           threads=12,
           tradeoff=0.7)
res.save('./models/res_western.zip')

In [ ]:
res = load_results('./models/res_western.zip')
growth_rates = res.growth_rates.copy()
growth_rates.taxon = growth_rates.taxon.astype(str)
growth_rates.sample_id = growth_rates.sample_id.astype(str)
exchanges = res.exchanges.copy()
exchanges.taxon = exchanges.taxon.astype(str)
annotations = res.annotations.copy()
annotations.metabolite = annotations.metabolite.astype(str)
res = GrowthResults(growth_rates=growth_rates, exchanges=exchanges, annotations=annotations)
interactions_res = interactions(results=res, threads=12, taxa=None)
interactions_res.to_csv('./models/interactions_western.csv')    

In [ ]:
mapping = pd.read_csv('./raw/DonorRecipientMapping.csv')
mapping['donor'] = mapping['donor'].str.split('_').str[0]
mapping['recipient'] = mapping['recipient'].str.split('_').str[0]

growth_results = load_results('./models/res_western.zip')
growth_rates = growth_results.growth_rates[growth_results.growth_rates['sample_id'].isin(pd.concat([mapping['donor'], mapping['recipient']]).unique())]

interaction_res = pd.read_csv('./models/interactions_western.csv')
interaction_res = interaction_res[interaction_res['sample_id'].isin(pd.concat([mapping['donor'], mapping['recipient']]).unique())]

met_int_dist_btw = {}
for row in mapping.iterrows():
    donor = row[1]['donor']
    recipient = row[1]['recipient']
    donor_df = interaction_res[interaction_res['sample_id'] == donor]
    donor_df['focal'] = donor_df['focal'].astype(str)
    donor_df['partner'] = donor_df['partner'].astype(str)
    recipient_df = interaction_res[interaction_res['sample_id'] == recipient]
    recipient_df['focal'] = recipient_df['focal'].astype(str)
    recipient_df['partner'] = recipient_df['partner'].astype(str)
    print(donor, recipient)
    all_taxa = pd.concat([donor_df['focal'], donor_df['partner'], recipient_df['focal'], recipient_df['partner']]).unique()
    if not donor_df.empty and not recipient_df.empty:
        dist_df = analysis.metabolic_interaction_distance(
            donor_df=donor_df,
            taxa=all_taxa,
            recipient_df=recipient_df,
            metabolites=None,
            method='mahalanobis'
        )
        dist_df['comparison'] = f"{donor} vs {recipient}"
        met_int_dist_btw[(donor, recipient)] = dist_df
    else:
        print(f"Donor {donor} or recipient {recipient} has no interaction data.")

nearest_neighbors = {}
for key, value in met_int_dist_btw.items():
    average_distance = []
    for row in value.iterrows():
        nearest_N = np.sort(row[1][:-1])[:10]
        mean_distance = np.mean(nearest_N)
        average_distance.append(mean_distance)
    results_df = pd.DataFrame({
        'donor': value.index,
        'average_distance': average_distance,
        'comparison': key[0] + ' vs ' + key[1]
    })
    nearest_neighbors[key] = results_df
nearest_neighbors_df = pd.concat(nearest_neighbors.values())
nearest_neighbors_df.rename(columns={'donor': 'taxon'}, inplace=True)
nearest_neighbors_df['sample_id'] = nearest_neighbors_df['comparison'].str.split(' vs ').str[0]
nearest_neighbors_df

In [ ]:
growth_rates.to_csv('./processed/growth_rates_t0.csv', index=False)
nearest_neighbors_df.to_csv('./processed/nearest_neighbors_t0.csv', index=False)

In [ ]:
abundance_simulated = pd.read_csv('./raw/SimulatedFMTCounts.csv')
taxonomy = pd.read_csv('./processed/full_taxonomy.csv')

abundance_simulated = (
    abundance_simulated
    .merge(
        taxonomy, 
        left_on='taxon', 
        right_on='taxon_id', 
        how='left'
    )
    .drop(columns=['strain'])
    .melt(
        id_vars=['taxon_id', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species'],
        var_name='sample_id',
        value_name='abundance'
    )
    .sort_values(by='abundance', ascending=False)
    .groupby(
        ['sample_id', 'species'],
        as_index=False
    )
    .agg(
        {
            'abundance': 'sum',
            **{col: 'first' for col in ['taxon_id', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']}
        }
    )
    .assign(taxon_id=lambda x: x['taxon_id'].astype(str))
    .rename(columns={'taxon_id': 'id'})
)
abundance_simulated

In [ ]:
metadata_simulated = build(abundance_simulated,
                           model_db='./raw/agora_species_103.qza',
                           out_folder='./models_simulated',
                           cutoff=0.0001,
                           threads=12)

In [ ]:
western = load_qiime_medium('./raw/western_diet_gut_agora.qza')
res_simulated = grow(manifest=metadata,
                     medium=western,
                     model_folder='./models_simulated',
                     threads=12,
                     tradeoff=0.7)
res_simulated.save('./models_simulated/res_western_simulated.zip')

In [ ]:
res_simulated = load_results('./models_simulated/res_western_simulated.zip')
growth_rates_simulated = res_simulated.growth_rates.copy()
growth_rates_simulated.taxon = growth_rates_simulated.taxon.astype(str)
growth_rates_simulated.sample_id = growth_rates_simulated.sample_id.astype(str)
exchanges_simulated = res_simulated.exchanges.copy()
exchanges_simulated.taxon = exchanges_simulated.taxon.astype(str)
annotations_simulated = res_simulated.annotations.copy()
annotations_simulated.metabolite = annotations_simulated.metabolite.astype(str)
res_simulated = GrowthResults(growth_rates=growth_rates_simulated, exchanges=exchanges_simulated, annotations=annotations_simulated)
interactions_res_simulated = interactions(results=res_simulated, threads=12, taxa=None)
interactions_res_simulated.to_csv('./models_simulated/interactions_western_simulated.csv')

## Incorporate Data From OpenBiome Project

In [ ]:
openbiome = pd.read_csv('./raw/OpenBiome_agora_mapped_counts.csv')
openbiome['taxon_id'] = openbiome['taxon_id'].astype(str)
openbiome = (
    openbiome
    .merge(
        reference_taxonomy_counts,
        left_on='taxon_id',
        right_on='NCBI Taxonomy ID',
        how='left',
    )
    .drop(columns=['RefSeq', 'organism_name', 'NCBI Taxonomy ID', 'id'])
)
data_cols = [col for col in openbiome.columns if col.startswith('SRR')]
openbiome[data_cols] = openbiome[data_cols].div(openbiome['genome_size_ungapped'], axis=0)
openbiome.drop(columns=['genome_size_ungapped'], inplace=True)
openbiome = (
    openbiome
    .melt(
        id_vars=['taxon_id', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species'],
        var_name='sample_id',
        value_name='abundance'
    )
    .rename(columns={'taxon_id': 'id'})
)
openbiome

In [ ]:
metadata = build(openbiome, 
                 model_db='./raw/agora_species_103.qza',
                 out_folder='./openbiome_models',
                 cutoff=0.0001,
                 threads=128)

In [ ]:
western = load_qiime_medium('./raw/western_diet_gut_agora.qza')
metadata = pd.read_csv('./openbiome_models/manifest.csv')
res = grow(manifest=metadata,
           medium=western,
           model_folder='./openbiome_models',
           threads=128,
           tradeoff=0.7)
res.save('./openbiome_models/res_western_openbiome.zip')

In [ ]:
res = load_results('./openbiome_models/res_western_openbiome.zip')
openbiome_interactions = interactions(
    results=res,
    threads=128,
    taxa=None
)
openbiome_interactions.to_csv('./openbiome_models/interactions_western_openbiome.csv')

In [ ]:
res_FMT = load_results('./models/res_western.zip')
openbiome_res = load_results('./openbiome_models/res_western_openbiome.zip')

interaction_FMT = pd.read_csv('./models/interactions_western.csv')
interaction_openbiome = pd.read_csv('./openbiome_models/res_interactions_openbiome.csv')

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def process_donor_recipient_pair(donor, recipient):
    try:
        donor_df = interaction_openbiome[interaction_openbiome['sample_id'] == donor].copy()
        donor_df['focal'] = donor_df['focal'].astype(str)
        donor_df['partner'] = donor_df['partner'].astype(str)

        recipient_df = interaction_FMT[interaction_FMT['sample_id'] == recipient].copy()
        recipient_df['focal'] = recipient_df['focal'].astype(str)
        recipient_df['partner'] = recipient_df['partner'].astype(str)

        # print(f"Processing {donor} vs {recipient}")

        if donor_df.empty or recipient_df.empty:
            print(f"Skipping {donor} vs {recipient} due to empty DataFrame")
            return None
        
        all_taxa = pd.concat([donor_df['focal'], donor_df['partner'], recipient_df['focal'], recipient_df['partner']]).unique()
        dist_df = analysis.metabolic_interaction_distance(
            donor_df=donor_df,
            taxa=all_taxa,
            recipient_df=recipient_df,
            metabolites=None,
            method='mahalanobis'
        )
        dist_df['comparison'] = f"{donor} vs {recipient}"
        return dist_df

    except Exception as e:
        print(f"Error processing {donor} vs {recipient}: {e}")
        return None
    
donors = interaction_openbiome['sample_id'].unique()
recipients = interaction_FMT[interaction_FMT['sample_id'].str.endswith('-0')]['sample_id'].unique()
pairs = list(product(donors, recipients))

with ProcessPoolExecutor(max_workers=128) as executor:
    future_to_pair = {executor.submit(process_donor_recipient_pair, donor, recipient): (donor, recipient) for donor, recipient in pairs}
    met_int_dist_btw = {}
    for future in as_completed(future_to_pair):
        pair = future_to_pair[future]
        try:
            result = future.result()
            if result is not None:
                met_int_dist_btw[pair] = result
        except Exception as e:
            print(f"Error processing pair {pair}: {e}")

nearest_neighbors = {}
for key, value in met_int_dist_btw.items():
    average_distance = []
    for row in value.iterrows():
        nearest_N = np.sort(row[1][:-1])[:10]
        mean_distance = np.mean(nearest_N)
        average_distance.append(mean_distance)
    results_df = pd.DataFrame({
        'donor': value.index,
        'average_distance': average_distance,
        'comparison': key[0] + ' vs ' + key[1]
    })
    nearest_neighbors[key] = results_df
nearest_neighbors_df = pd.concat(nearest_neighbors.values())
nearest_neighbors_df.rename(columns={'donor': 'taxon'}, inplace=True)
nearest_neighbors_df['sample_id'] = nearest_neighbors_df['comparison'].str.split(' vs ').str[0]
nearest_neighbors_df

In [ ]:
# calculate all metabolic interaction distances between openbiome donors and FMT recipients
openbiome_donor_growth = openbiome_res.growth_rates.copy()
openbiome_donor_growth.taxon = openbiome_donor_growth.taxon.astype(str)

donors = openbiome_donor_growth['sample_id'].unique()
recipients = res_FMT.growth_rates[res_FMT.growth_rates['sample_id'].str.endswith('-0')]['sample_id'].unique()

met_int_dist_btw = {}
for donor, recipient in product(donors, recipients):
    donor_df = interaction_openbiome[interaction_openbiome['sample_id'] == donor]
    donor_df['focal'] = donor_df['focal'].astype(str)
    donor_df['partner'] = donor_df['partner'].astype(str)
    recipient_df = interaction_FMT[interaction_FMT['sample_id'] == recipient]
    recipient_df['focal'] = recipient_df['focal'].astype(str)
    recipient_df['partner'] = recipient_df['partner'].astype(str)
    print(donor, recipient)
    all_taxa = pd.concat([donor_df['focal'], donor_df['partner'], recipient_df['focal'], recipient_df['partner']]).unique()
    if not donor_df.empty and not recipient_df.empty:
        dist_df = analysis.metabolic_interaction_distance(
            donor_df=donor_df,
            taxa=all_taxa,
            recipient_df=recipient_df,
            metabolites=None,
            method='mahalanobis'
        )
        dist_df['comparison'] = f"{donor} vs {recipient}"
        met_int_dist_btw[(donor, recipient)] = dist_df
    else:
        print(f"Donor {donor} or recipient {recipient} has no interaction data.")

nearest_neighbors = {}
for key, value in met_int_dist_btw.items():
    average_distance = []
    for row in value.iterrows():
        nearest_N = np.sort(row[1][:-1])[:10]
        mean_distance = np.mean(nearest_N)
        average_distance.append(mean_distance)
    results_df = pd.DataFrame({
        'donor': value.index,
        'average_distance': average_distance,
        'comparison': key[0] + ' vs ' + key[1]
    })
    nearest_neighbors[key] = results_df
nearest_neighbors_df = pd.concat(nearest_neighbors.values())
nearest_neighbors_df.rename(columns={'donor': 'taxon'}, inplace=True)
nearest_neighbors_df['sample_id'] = nearest_neighbors_df['comparison'].str.split(' vs ').str[0]
nearest_neighbors_df

In [ ]:
# calculate SCFA production for all openbiome donors
metabolites = {
    'Formate': 'for[e]', 
    'acetate': 'ac[e]', 
    'propionate': 'ppa[e]', 
    'butyrate': 'but[e]', 
    'Valeric Acid': 'M03134[e]', 
    'Sulfite': 'so3[e]', 
    'thiosulfate(2-)': 'tsul[e]', 
    'Methanethiol': 'ch4s[e]',
    'Hydrogen': 'h2[e]', 
}

ex_metabolites = openbiome_res.exchanges[openbiome_res.exchanges.metabolite.isin(metabolites.values())].copy()
ex_metabolites = ex_metabolites.query('taxon != "medium" and direction == "export"')
ex_metabolites['flux'] = ex_metabolites['flux'] * ex_metabolites['abundance']
ex_metabolites = (
    ex_metabolites
    .groupby(['sample_id', 'metabolite'], as_index=False)
    .agg({'flux': 'sum'})
)

unique_samples = ex_metabolites['sample_id'].unique()
unique_metabolites = list(metabolites.values())

template = pd.MultiIndex.from_product(
    [unique_samples, unique_metabolites],
    names=['sample_id', 'metabolite']
).to_frame(index=False)

ex_metabolites = (
    template
    .merge(ex_metabolites[['metabolite', 'sample_id', 'flux']], 
           on=['sample_id', 'metabolite'], 
           how='left')
    .fillna(0)
)

id2metabolites = {v: k for k, v in metabolites.items()}
ex_metabolites['metabolite'] = ex_metabolites['metabolite'].map(id2metabolites)
ex_metabolites

In [ ]:
# calculate simulated engraftment measured in bray curtis dissimilarity between a donor and a recipient
from skbio.diversity import beta_diversity

recipient_abundance = res_FMT.growth_rates[res_FMT.growth_rates['sample_id'].str.endswith('-6')][['sample_id', 'taxon', 'abundance']]
recipient_abundance['taxon'] = recipient_abundance['taxon'].astype(str)
donor_abundance = openbiome_res.growth_rates[['sample_id', 'taxon', 'abundance']].copy()
donor_abundance['taxon'] = donor_abundance['taxon'].astype(str)

abundance_all = (
    pd.concat([recipient_abundance, donor_abundance])
    .pivot(index='sample_id', columns='taxon', values='abundance')
    .fillna(0)
)

bc_dm = beta_diversity(
    metric='braycurtis',
    counts=abundance_all.values,
    ids=abundance_all.index
)
distMatrix = pd.DataFrame(bc_dm.data, index=bc_dm.ids, columns=bc_dm.ids)
distMatrix

In [ ]:
abundance = pd.read_csv('./raw/SimulatedFMTCounts_OpenBiome.csv')
taxonomy = pd.read_csv('./processed/full_taxonomy.csv')
abundance.set_index('taxon', inplace=True)
abundance.columns = abundance.columns.str.replace(' ', '_')
abundance = (
    abundance
    .reset_index()
    .merge(
        taxonomy,
        left_on='taxon',
        right_on='taxon_id',
        how='left'
    )
    .drop(columns=['strain', 'taxon_id'])
    .melt(
        id_vars=['taxon', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species'],
        var_name='sample_id',
        value_name='abundance'
    )
    .sort_values(by='abundance', ascending=False)
    .assign(taxon=lambda x: x['taxon'].astype(str))
    .rename(columns={'taxon': 'id'})
)
abundance

In [ ]:
metadata_simulated = build(abundance,
                           model_db='./raw/agora_species_103.qza',
                           out_folder='/data/zetianzhang/models/',
                           cutoff=0.0001,
                           threads=96)

In [2]:
metadata_simulated = pd.read_csv('/data/zetianzhang/models/manifest.csv')
western = load_qiime_medium('./raw/western_diet_gut_agora.qza')
res_simulated = grow(manifest=metadata_simulated,
                     medium=western,
                     threads=120,
                     tradeoff=0.7,
                     model_folder='/data/zetianzhang/models/')
res_simulated.save('./openbiome_simulated_models/res_western_simulated_openbiome.zip')

In [ ]:
res_simulated.exchanges.to_csv('./openbiome_simulated_models/exchanges_western_simulated_openbiome.csv')
res_simulated.growth_rates.to_csv('./openbiome_simulated_models/growth_western_simulated_openbiome.csv')
res_simulated.annotations.to_csv('./openbiome_simulated_models/annotations_western_simulated_openbiome.csv')

In [7]:
from zipfile import ZipFile

with ZipFile('./openbiome_simulated_models/res_western_simulated_openbiome.zip', 'w') as zf:
    zf.write('./openbiome_simulated_models/exchanges_western_simulated_openbiome.csv', arcname='exchanges.csv')
    zf.write('./openbiome_simulated_models/growth_western_simulated_openbiome.csv', arcname='growth_rates.csv')
    zf.write('./openbiome_simulated_models/annotations_western_simulated_openbiome.csv', arcname='annotations.csv')